# 🏛️ oracc-parser — Quickstart

This notebook shows you the actual data available, how to parse it, and where everything lives on disk.

In [1]:
# Install oracc-parser from PyPI
!pip install oracc-parser -q

# --- Optional: persist downloaded data across Colab sessions using Google Drive ---
# Without this, data downloads to /content/oracc_data and is lost when the session ends.
# Uncomment the lines below to mount your Drive and store data there persistently.
#
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ["ORACC_DATA_DIR"] = "/content/drive/MyDrive/oracc_data"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 3.0 MB/s eta 0:00:00
Mounted at /content/drive


## 🎯 Goals of this notebook

1. **Get the catalogue data** — download catalogues (metadata) from Zenodo
2. **Parse a project** — load word CSVs, inspect the project catalogue, and explore individual tablets
3. **Get flat DataFrames** — metadata, transliterations, and full combined tables ready for analysis
4. **Export** — save results as CSV or JSONL

## 1. Get the catalogue data (Zenodo)

Download project catalogues from Zenodo (doi: 10.5281/zenodo.20625379). These contain all the metadata stored per text by project.

In [2]:
import os
from pathlib import Path
from oracc_parser.settings import DATA_DIR, OUTPUT_DIR, CACHE_DIR, JSONZIP_DIR, CATALOGUE_DIR
from oracc_parser.download.fetch_data import fetch_data

# Only catalogues are required upfront; word CSVs are fetched per-project on demand.
catalogues_ok = CATALOGUE_DIR.exists() and any(CATALOGUE_DIR.glob("*.csv"))

if not catalogues_ok:
    print("Downloading catalogues from Zenodo...")
    fetch_data()
else:
    print(f"Catalogues found in {CATALOGUE_DIR} ({len(list(CATALOGUE_DIR.glob('*.csv')))} projects)")

Fetching file list from https://zenodo.org/records/20625379...
  catalogues.zip (2.7 MB)


catalogues.zip: 100%|██████████| 2.78M/2.78M [00:03<00:00, 852kB/s] 



📂 Extracting catalogues.zip ...


Extracting catalogues: 100%|██████████| 138/138 [00:01<00:00, 73.26file/s]

✅ Extracted 138 files to /content/drive/MyDrive/oracc_data/catalogues

✓ Done. Data is ready in /content/drive/MyDrive/oracc_data


## 2. Parse a project

### How project data is organized on Zenodo

ORACC projects follow a two-level hierarchy: an **umbrella project** (e.g. `saao`) groups related **sub-projects** (e.g. `saao/saa01`, `saao/saa02`, …). On Zenodo, word CSVs are stored as one zip per umbrella — so `saao.zip` contains the data for all 22 State Archives of Assyria volumes.

When you request a sub-project for the first time, the package downloads the entire umbrella zip, extracts all sub-projects it contains, and caches them locally. Subsequent requests for any project in the same umbrella (e.g. switching from `saao/saa01` to `saao/saa02`) are instant — no further download needed.

A few projects have no umbrella (e.g. `babcity`, `balt`, and many more) and are stored as their own single-project zip.

For demonstration purposes, `babcity` was picked below as it is a small project with fast download.

**Change the variable `PROJECT` to any valid oracc project or subproject name to test with different projects!**

In [3]:
from oracc_parser import parse_project_from_word_csvs, RunConfig, get_metadata_table
from oracc_parser.io.word_csv import load_word_csvs_from_dir
from oracc_parser.settings import WORD_CSV_DIR

# Change to any project available in oracc_csvs/
PROJECT = "babcity"
LIMIT = 5  # Set to None to load all tablets

project_slug = PROJECT.replace("/", "-")
project_csv_dir = WORD_CSV_DIR / project_slug

# Loads from disk if already downloaded; downloads from Zenodo otherwise
word_dfs = load_word_csvs_from_dir(project_csv_dir, project=PROJECT)
if LIMIT:
    word_dfs = dict(list(word_dfs.items())[:LIMIT])

config = RunConfig(fetch_translations=False)  # enable after downloading translations from Zenodo
records = parse_project_from_word_csvs(PROJECT, word_dfs, config=config)
print(f"Loaded {len(records)} tablets from {PROJECT}")

display(get_metadata_table(records))

babcity.zip: 100%|██████████| 1.01M/1.01M [00:01<00:00, 750kB/s]


  -> Extracted babcity projects to /content/drive/MyDrive/oracc_data/oracc_csvs


11:05:23 | INFO    | oracc_parser | Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/babcity
INFO:oracc_parser:Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/babcity
Processing babcity:   0%|          | 0/5 [00:00<?, ?it/s]11:05:23 | ERROR   | oracc_parser | Error loading language qcu-949: 1 validation error for Language
normalized_language
  Input should be 'Sumerian', 'Proto-cuneiform', 'Akkadian', 'Ugaritic', 'Urartian', 'Persian', 'Hittite', 'Aramaic', 'Elamite', 'Hurrian', 'Canaanite', 'Numbers', 'Unclear', 'Proto-Elamite', 'Greek' or 'Egyptian' [type=literal_error, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.12/v/literal_error
ERROR:oracc_parser:Error loading language qcu-949: 1 validation error for Language
normalized_language
  Input should be 'Sumerian', 'Proto-cuneiform', 'Akkadian', 'Ugaritic', 'Urartian', 'Persian', 'Hittite', 'Aramaic', 'Elamite', 'Hurrian', 'Canaan

Loaded 5 tablets from babcity


,id,project,text_id,genre,archive,provenance,pleiades_id,state_supergroup,period,start_year,end_year,accession_museum_publication_numbers,secondary_literature,credits,cite_as
0,babcity_P261352,babcity,P261352,Legal,Murašû Archive,Nippur,912910,Mix,achaemenid,-547,-331,"BE 10, 056; CBS 05160",Clay Aram.Endors. 306 #17(Aram.only); PBS 02/1...,Based on a transliteration by Heather D. Baker...,Please cite this page as http://oracc.org/babc...
1,babcity_P285916,babcity,P285916,Legal,Egibi Archive,Babylon,893951,Mix,achaemenid,-547,-331,"Rental 130; Sp 2, 0016; BM 034544","Zawadzki, Rental of Houses 130","Based on the edition of Stefan Zawadzki, The R...",Please cite this page as http://oracc.org/babc...
2,babcity_P295135,babcity,P295135,Legal,Ea-ilutu-bani Archive,Borsippa,893964,Mix,achaemenid,-547,-331,BRM 1 68; MLC 330,"Joannes,Archives de Borsippa 94 (Tr) and 325 (T)","Adapted, lemmatised and translated by Lindsey ...",Please cite this page as http://oracc.org/babc...
3,babcity_P295289,babcity,P295289,Legal,Ea-ilutu-bani Archive,Borsippa,893964,Mix,Neo-Babylonian,-626,-539,BRM 1 58; MLC 00491,"Joannes, Archives de Borsippa 113 (Tr) and 323(T)","Adapted, lemmatised and translated by Lindsey ...",Please cite this page as http://oracc.org/babc...
4,babcity_P296350,babcity,P296350,Legal,Ea-ilutu-bani Archive,Borsippa,893964,Mix,Neo-Babylonian,-626,-539,BRM 1 43; MLC 01668,,"Adapted, lemmatised and translated by Lindsey ...",Please cite this page as http://oracc.org/babc...


#### View one text

The texts were processed from the json format to a word-level DataFrame (see notebook 4 for more details on the process).

In [4]:
from oracc_parser.io.word_csv import record_to_word_dataframe

word_df = record_to_word_dataframe(records[0])
print(f"Word-level CSV for {records[0].metadata.id_text}: {len(word_df)} rows")
display(word_df.head(10))

Word-level CSV for P261352: 88 rows


,text_id,project,word_index,frag,ref,inst,form,lemma_form,sense,norm,raw_pos,lang,line,signs_reading,signs_break_pct,unicode,break_info
0,P261352,babcity,0,1,P261352.3.1,n,1,,,,n,akk-x-neobab,3,[1],1.00,𒁹,missing
1,P261352,babcity,1,1/2,P261352.3.2,n,1/2,,,,n,akk-x-neobab,3,[1/2],1.00,𒈦,missing
2,P261352,babcity,2,MA.NA,P261352.3.3,manû[unit]N,MA.NA,manû,unit,manû,N,akk-x-neobab,3,[MA].[NA],1.00,𒈠; 𒈾,missing; missing
3,P261352,babcity,3,KU₃.BABBAR,P261352.3.4,kaspi[silver]N,KU₃.BABBAR,kaspu,silver,kaspi,N,akk-x-neobab,3,[KU₃].[BABBAR],1.00,𒆬; 𒌓,missing; missing
4,P261352,babcity,4,i-di,P261352.3.5,+idu[arm//rent]N'N$idī,i-di,idu,rent,idī,N,akk-x-neobab,3,[i-di],1.00,𒄿; 𒁲,missing; missing
5,P261352,babcity,5,E₂,P261352.3.6,bīti[house]N,E₂,bītu,house,bīti,N,akk-x-neobab,3,[E₂],1.00,𒂍,missing
6,P261352,babcity,6,ša₂,P261352.3.7,ša[of]DET,ša₂,ša,of,ša,DET,akk-x-neobab,3,[ša₂],1.00,𒃻,missing
7,P261352,babcity,7,MU,P261352.3.8,šatti[year]N,MU,šattu,year,šatti,N,akk-x-neobab,3,[MU],1.00,𒈬,missing
8,P261352,babcity,8,1-KAM],P261352.3.9,n,1-KAM,,,,n,akk-x-neobab,3,[1-KAM],1.00,𒁹; 𒄰,missing; missing
9,P261352,babcity,9,{m}da-ri-ia-⸢a⸣-muš,P261352.4.1,Darius[1]RN,{m}da-ri-ia-a-muš,Darius,1,Darius,RN,akk-x-neobab,4,{m}da-ri-ia-⸢a⸣-muš,0.08,𒁹; 𒁕; 𒊑; 𒅀; 𒀀; 𒈲,complete; complete; complete; complete; damage...


### 2.1 Available projects and bulk download

The table below lists every project in the dataset, how many texts it contains, and whether the word CSVs are already downloaded locally. The  column is empty until you run the cell below it.


In [5]:
import pandas as pd
from oracc_parser.settings import CATALOGUE_DIR, WORD_CSV_DIR

rows = []
for cat_file in sorted(CATALOGUE_DIR.glob("*.csv")):
    slug = cat_file.stem
    project = slug.replace("-", "/", 1)
    umbrella = slug.split("-")[0]
    n_texts = sum(1 for _ in open(cat_file, encoding="utf-8")) - 1  # fast line count
    downloaded = (WORD_CSV_DIR / slug).exists() and any((WORD_CSV_DIR / slug).glob("*.csv"))
    rows.append({"project": project, "umbrella": umbrella, "texts": n_texts, "downloaded": "✓" if downloaded else ""})

available = pd.DataFrame(rows)
n_dl = (available["downloaded"] == "✓").sum()
print(f"{len(available)} projects, {available['texts'].sum()} texts total — {n_dl} already downloaded")
display(available)


138 projects, 55393 texts total — 1 already downloaded


,project,umbrella,texts,downloaded
0,adsd/adart1,adsd,417,
1,adsd/adart2,adsd,417,
2,adsd/adart3,adsd,417,
3,adsd/adart5,adsd,106,
4,adsd/adart6,adsd,180,
...,...,...,...,...
133,tcma/taban,tcma,14,
134,tcma/tsa1,tcma,17,
135,tcma/tsh1,tcma,262,
136,tcma/ugarit,tcma,5,


To download word CSVs for all projects at once, run the cell below. Projects are grouped by umbrella on Zenodo, so each download fetches an entire group at once — e.g. downloading  also retrieves all other  volumes in the same request.

Run time: 20-25 minutes

In [ ]:
# from collections import defaultdict
# from tqdm import tqdm
# from oracc_parser.download.fetch_data import extract_project_csvs

# # Group catalogue entries by umbrella
# umbrellas = defaultdict(list)
# for slug in sorted(f.stem for f in CATALOGUE_DIR.glob("*.csv")):
#     umbrellas[slug.split("-")[0]].append(slug)

# # Only download umbrella groups that are not fully on disk yet
# to_download = [
#     (umbrella, slugs) for umbrella, slugs in sorted(umbrellas.items())
#     if not all(
#         (WORD_CSV_DIR / s).exists() and any((WORD_CSV_DIR / s).glob("*.csv"))
#         for s in slugs
#     )
# ]

# if not to_download:
#     print("All projects already downloaded.")
# else:
#     print(f"Downloading {len(to_download)} umbrella group(s) covering "
#           f"{sum(len(s) for _, s in to_download)} project(s)...")
#     skipped = []
#     for umbrella, slugs in tqdm(to_download, unit="group"):
#         try:
#             extract_project_csvs(slugs[0].replace("-", "/", 1))
#         except (FileNotFoundError, ValueError) as e:
#             skipped.append((umbrella, str(e).split(":")[0]))
#     if skipped:
#         print(f"\nSkipped {len(skipped)} group(s) with no downloadable word CSVs:")
#         for name, reason in skipped:
#             print(f"  {name}: {reason}")
#     print("Done. Re-run the cell above to see updated download status.")

## 3. Get flat DataFrames (for analysis & export)

The word-level DataFrames can then be processed to represent the text in transliteration, normalization, or Unicode cuneiform. Translation is also possible - ==see notebook 4==.

In [7]:
# The text in different representations
tablet = records[0]
c = tablet.content

print("TRANSLITERATION:")
print(f"{c.transliterated_str_representation.text[:200] if c.transliterated_str_representation else '(none)'}")
print()
print("NORMALIZATION:")
print(f"{c.normalized_str_representation.text[:200] if c.normalized_str_representation else '(none)'}")
print()
print("UNICODE CUNEIFORM:")
print(f"{c.unicode_str_representation.text[:200] if c.unicode_str_representation else '(none)'}")
print()
# print("ENGLISH TRANSLATION:")
# print(f"{c.english_translation[:200] if c.english_translation else '(none)'}")

TRANSLITERATION:
1 1/2 MA.NA KU₃.BABBAR i-di E₂ ša₂ MU 1-KAM]
{m}da-ri-ia-⸢a⸣-muš LUGAL ša₂ {[m}{d}EN-it-tan-nu]
A ša₂ {m}{d}EN.LIL₂-TIN{iṭ} ša₂ ina IGI {m}ti-ri-ra-ka-am-ma
DUMU E₂ ša₂ {m}{d}EN.LIL₂-MU-MU
{m}{d}MAŠ-Š

NORMALIZATION:
UNK UNK manû kaspi idī bīti ša šatti UNK
Darius šarri ša Bel-ittannu
māru ša Enlil-uballiṭ ša ina pāni Tirirakamma
mār bīti ša Enlil-šum-iddin
Ninurta-ahu-uṣur ardu ša Bel-ittannu
ina qāt Tirirakamma


UNICODE CUNEIFORM:
𒁹 𒈦 𒈠𒈾 𒆬𒌓 𒄿𒁲 𒂍 𒃻 𒈬 𒁹𒄰
𒁹𒁕𒊑𒅀𒀀𒈲 𒈗 𒃻 𒁹𒀭𒂗𒀉𒆗𒉡
𒀀 𒃻 𒁹𒀭𒂗𒆤𒁷𒀉 𒃻 𒀸 𒅆 𒁹𒋾𒊑𒊏𒅗𒄠𒈠
𒌉 𒂍 𒃻 𒁹𒀭𒂗𒆤𒈬𒈬
𒁹𒀭𒈦𒋀𒌶 𒇽𒀴 𒃻 𒁹𒀭𒂗𒀉𒆗𒉡
𒀸 𒋗𒈫 𒁹𒋾𒊑𒊏𒅗𒄠𒈠
𒈠𒆟 𒂊𒋛𒀀 𒌑𒃻𒊍𒍝𒊍𒄠𒈠
𒁹𒀭𒈦𒋀𒌶 𒃻 𒆬𒌓 𒀪
𒁹 𒈦 𒈠𒈾 𒄿𒁲 𒂍 𒃻 𒈬 𒁹𒄰
𒀉𒋾 𒁹𒀭𒂗𒀉𒆗𒉡
𒀀𒈾 𒁹𒋾𒊑𒊏𒅗𒄠𒈠
𒀸𒀭𒁷
𒇽𒈬𒆥 𒁹𒀭𒈦𒂵𒅖 𒀀 𒃻 𒁹𒋺𒆗𒉡




The texts can be downloaded using the function `get_full_flat_table`. It combines the processed documents with the simplified metadata.

In [8]:
from oracc_parser import get_metadata_table, get_transliterations, get_full_flat_table

# Full flat table — everything in one DataFrame
flat = get_full_flat_table(records)
print(f"Full table: {flat.shape[0]} rows × {flat.shape[1]} columns")
print(f"Columns: {list(flat.columns)}")
display(flat)

Full table: 5 rows × 16 columns
Columns: ['id', 'project', 'text_id', 'genre', 'archive', 'provenance', 'period', 'start_year', 'end_year', 'transliteration', 'normalization', 'lemmatization', 'unicode', 'translation', 'total_tokens', 'tokens_without_broken']


,id,project,text_id,genre,archive,provenance,period,start_year,end_year,transliteration,normalization,lemmatization,unicode,translation,total_tokens,tokens_without_broken
0,babcity_P261352,babcity,P261352,Legal,Murašû Archive,Nippur,achaemenid,-547,-331,1 1/2 MA.NA KU₃.BABBAR i-di E₂ ša₂ MU 1-KAM]\n...,UNK UNK manû kaspi idī bīti ša šatti UNK\nDari...,UNK UNK manû kaspu idu bītu ša šattu UNK\nDari...,𒁹 𒈦 𒈠𒈾 𒆬𒌓 𒄿𒁲 𒂍 𒃻 𒈬 𒁹𒄰\n𒁹𒁕𒊑𒅀𒀀𒈲 𒈗 𒃻 𒁹𒀭𒂗𒀉𒆗𒉡\n𒀀 𒃻 ...,,88,88
1,babcity_P285916,babcity,P285916,Legal,Egibi Archive,Babylon,achaemenid,-547,-331,E₂ ša₂ {m}ni-din-ti-{d}EN DUMU {m}ši-rik\nša₂ ...,bītu ša Nidinti-Bel māri Širku\nša ṭāh bīt Suq...,bītu ša Nidinti-Bel māru Širku\nša ṭāh bītu Su...,𒂍 𒃻 𒁹𒉌𒁷𒋾𒀭𒂗 𒌉 𒁹𒅆𒋆\n𒃻 𒁕 𒂍 𒁹𒋻𒀀𒀀\n𒁹𒉌𒁷𒌈 𒌉 𒁹𒅆𒋆 𒀀 𒁹𒂊𒐕...,,92,92
2,babcity_P295135,babcity,P295135,Legal,Ea-ilutu-bani Archive,Borsippa,achaemenid,-547,-331,E₂-⸢su⸣ ša₂ {e₂}a-su-up ša₂ {m}⸢x⸣-[...]\nA-šu...,bīssu ša asup ša X\nmārišu ša Luṣi-ana-nur-Mar...,bītu ša asuppu ša X\nmāru ša Luṣi-ana-nur-Mard...,𒂍𒋢 𒃻 𒂍𒀀𒋢𒌒 𒃻 𒁹Xx\n𒀀𒋙 𒃻 𒁹𒇻𒍮𒀀𒈾𒂟𒀭𒀫𒌓 x\n𒀀𒈾 𒈬𒀭𒈾 𒈫 𒂆 ...,,84,81
3,babcity_P295289,babcity,P295289,Legal,Ea-ilutu-bani Archive,Borsippa,Neo-Babylonian,-626,-539,6 GI-MEŠ KI{ti₃} E₂ {d}nam-tar-ri\nša₂ qe₂-reb...,UNK qanû erṣet bīt Namtar\nša qereb Borsippa š...,UNK qanû erṣetu bītu Namtar\nša qerbu Borsippa...,𒐋 𒄀𒈨𒌍 𒆠𒁴 𒂍 𒀭𒉆𒋻𒊑\n𒃻 𒆠𒆗 𒁈𒉺𒇻𒆠 𒃻 𒁹𒈾𒁲𒉡\n𒀀𒋙 𒃻 𒁹𒇻𒍪𒀀𒈾𒂟...,,89,89
4,babcity_P296350,babcity,P296350,Legal,Ea-ilutu-bani Archive,Borsippa,Neo-Babylonian,-626,-539,E₂ ša₂ {f}GEME₂-{d}su-ti-ti DUMU.MI₃-su\nša₂ {...,bītu ša Amat-Sutiti mārassu\nša Iddinaya mār R...,bītu ša Amat-Sutiti mārtu\nša Iddinaya māru Ra...,𒂍 𒃻 𒊩𒊩𒆳𒀭𒋢𒋾𒋾 𒌉𒈨𒋢\n𒃻 𒁹𒋧𒈾𒀀 𒀀 𒇽𒃲𒀀𒋛𒂊\n𒀀𒈾 𒄿𒁲 𒂍 𒀸 𒅆 𒁹...,,96,95


## 4. Export & see where output goes

The table above can be saved locally in as a JSONL or CSV format. The output directory of this package is under `enriched_data`. If you want to change it to another directory, change the value of the variable `out_dir`.

In [9]:
from oracc_parser.settings import OUTPUT_DIR

out_dir = OUTPUT_DIR
out_dir.mkdir(parents=True, exist_ok=True)

jsonl_path = out_dir / f"{PROJECT.replace('/', '_')}.jsonl"
csv_path   = out_dir / f"{PROJECT.replace('/', '_')}.csv"

flat.to_json(jsonl_path, orient="records", lines=True, force_ascii=False)
flat.to_csv(csv_path, index=False)

print(f"Exported to:")
print(f"  JSONL: {jsonl_path}  ({jsonl_path.stat().st_size / 1024:.1f} KB)")
print(f"  CSV:   {csv_path}  ({csv_path.stat().st_size / 1024:.1f} KB)")
print(f"\nOutput directory: {out_dir}")

Exported to:
  JSONL: /content/drive/MyDrive/oracc_data/output/babcity.jsonl  (17.4 KB)
  CSV:   /content/drive/MyDrive/oracc_data/output/babcity.csv  (16.1 KB)

Output directory: /content/drive/MyDrive/oracc_data/output


## What's next?

- **Explore what's in the dataset:** See notebook `02_reference_data.ipynb` for a full overview of collected projects, catalogues, provenance, periods, and other reference data.
- **Configure parsing:** See notebook `03_configure_and_export.ipynb` for masking PNs, dropping broken signs, and changing output formats.
- **understand the process:** see notebook `04_oracc_json_processing.ipynb` for in-depth explanations on how the package processes the original oracc files and metadata.